# 10 — Degerlendirme v2
Fine-tuned MobileNetV3 vs v1 modelleri. Kritik kontrol: over-prediction duzeltildi mi?

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from sklearn.metrics import f1_score, hamming_loss
from torch.utils.data import DataLoader

# --- Yerel ---
CODE_ROOT = Path('../')
DATA_ROOT = Path('../')

# --- Colab ---
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT = Path('/content/drive/MyDrive/film-genre-project')
# DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')
# import sys; sys.path.insert(0, str(CODE_ROOT / 'src'))

import sys
sys.path.insert(0, str(CODE_ROOT / 'src'))

from dataset import PosterDataset
from eval import find_best_thresholds, evaluate
from model import PosterCNN
from transforms import val_transforms

POSTERS_DIR   = DATA_ROOT / 'posters'
TEST_CSV      = DATA_ROOT / 'test.csv'
VAL_CSV       = DATA_ROOT / 'val.csv'
MLB_PKL       = DATA_ROOT / 'mlb.pkl'
FINETUNE_CKPT = CODE_ROOT / 'checkpoints' / 'finetune' / 'stage3_best.pt'
SCRATCH_CKPT  = CODE_ROOT / 'checkpoints' / 'scratch'  / 'best.pt'
BASELINE_CKPT = CODE_ROOT / 'checkpoints' / 'baseline' / 'best.pt'

with open(MLB_PKL, 'rb') as f:
    mlb = pickle.load(f)
N_CLASSES = len(mlb.classes_)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  N_CLASSES: {N_CLASSES}')

In [ ]:
BATCH_SIZE  = 64
NUM_WORKERS = 2

val_ds  = PosterDataset(VAL_CSV,  POSTERS_DIR, mlb, transform=val_transforms)
test_ds = PosterDataset(TEST_CSV, POSTERS_DIR, mlb, transform=val_transforms)

val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Val: {len(val_ds):,}  Test: {len(test_ds):,}')

## 1. Fine-tuned MobileNetV3 (v2)

In [ ]:
weights = torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
ft_model = torchvision.models.mobilenet_v3_small(weights=None)
ft_model.classifier[-1] = nn.Linear(ft_model.classifier[-1].in_features, N_CLASSES)
ft_model.load_state_dict(torch.load(FINETUNE_CKPT, map_location=DEVICE))
ft_model = ft_model.to(DEVICE)
ft_model.eval()
print('Fine-tuned model yuklendi:', FINETUNE_CKPT)

ft_thresholds = find_best_thresholds(ft_model, val_loader, DEVICE, N_CLASSES)
ft_metrics    = evaluate(ft_model, test_loader, DEVICE, ft_thresholds)

print('\n=== Fine-tuned MobileNetV3 — Test Metrikleri ===')
print(f'  Macro F1    : {ft_metrics["macro_f1"]:.4f}')
print(f'  Micro F1    : {ft_metrics["micro_f1"]:.4f}')
print(f'  Hamming Loss: {ft_metrics["hamming_loss"]:.4f}')
print('\nOptimize edilmis threshold|Per-class F1:')
for g, t, f in zip(mlb.classes_, ft_thresholds, ft_metrics['per_class_f1']):
    print(f'  {g:20s}: threshold={t:.2f}  F1={f:.4f}')

## 2. Kritik Kontrol — Over-prediction Duzeltildi mi?

In [ ]:
def avg_predictions_per_film(mdl, loader, thresholds):
    mdl.eval()
    thresh_t = torch.tensor(thresholds, device=DEVICE)
    counts = []
    with torch.no_grad():
        for imgs, _ in loader:
            probs = torch.sigmoid(mdl(imgs.to(DEVICE)))
            preds = (probs > thresh_t).sum(dim=1).cpu().numpy()
            counts.extend(preds.tolist())
    counts = np.array(counts)
    return counts.mean(), counts.std(), np.bincount(counts.astype(int), minlength=10)

mean_preds, std_preds, hist = avg_predictions_per_film(ft_model, test_loader, ft_thresholds)
print(f'Fine-tuned model — ortalama tahmin/film: {mean_preds:.2f} (std={std_preds:.2f})')
print('Dagilim:')
for n_genres, count in enumerate(hist[:8]):
    bar = '#' * (count // 10)
    print(f'  {n_genres} tur: {count:4} film  {bar}')

## 3. v1 Modelleri Yukle (karsilastirma icin)

In [ ]:
scratch_metrics   = None
baseline_metrics  = None

if SCRATCH_CKPT.exists():
    scratch_model = PosterCNN(n_classes=N_CLASSES).to(DEVICE)
    scratch_model.load_state_dict(torch.load(SCRATCH_CKPT, map_location=DEVICE))
    scratch_model.eval()
    scratch_th      = find_best_thresholds(scratch_model, val_loader, DEVICE, N_CLASSES)
    scratch_metrics = evaluate(scratch_model, test_loader, DEVICE, scratch_th)
    print(f'Scratch CNN  — Macro F1: {scratch_metrics["macro_f1"]:.4f}')

if BASELINE_CKPT.exists():
    bl_model = torchvision.models.mobilenet_v3_small(weights=None)
    bl_model.classifier[-1] = nn.Linear(bl_model.classifier[-1].in_features, N_CLASSES)
    bl_model.load_state_dict(torch.load(BASELINE_CKPT, map_location=DEVICE))
    bl_model = bl_model.to(DEVICE)
    bl_model.eval()
    bl_th            = find_best_thresholds(bl_model, val_loader, DEVICE, N_CLASSES)
    baseline_metrics = evaluate(bl_model, test_loader, DEVICE, bl_th)
    print(f'Baseline v1  — Macro F1: {baseline_metrics["macro_f1"]:.4f}')

print(f'Fine-tuned v2 — Macro F1: {ft_metrics["macro_f1"]:.4f}')

## 4. Karsilastirma Grafikleri

In [ ]:
genres = list(mlb.classes_)
x = np.arange(len(genres))
w = 0.25

fig, ax = plt.subplots(figsize=(15, 6))

if scratch_metrics:
    ax.bar(x - w, scratch_metrics['per_class_f1'],  w, label='Scratch CNN v1',    alpha=0.8, color='#3498db')
if baseline_metrics:
    ax.bar(x,     baseline_metrics['per_class_f1'], w, label='Baseline v1 (frozen)', alpha=0.8, color='#e67e22')
ax.bar(x + w,     ft_metrics['per_class_f1'],       w, label='Fine-tuned v2',     alpha=0.8, color='#2ecc71')

ax.set_xticks(x)
ax.set_xticklabels(genres, rotation=45, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-class F1 — Tum Modeller')
ax.legend()
plt.tight_layout()
plt.show()

# Ozet tablo
print('\n=== Ozet ===')
print(f'{"Model":<25} {"Macro F1":>10} {"Micro F1":>10} {"Hamming":>10} {"Ort.Pred/film":>15}')
print('-' * 75)
if scratch_metrics:
    sm, _ ,sh = avg_predictions_per_film(scratch_model, test_loader, scratch_th)
    print(f'{"Scratch CNN v1":<25} {scratch_metrics["macro_f1"]:>10.4f} {scratch_metrics["micro_f1"]:>10.4f} {scratch_metrics["hamming_loss"]:>10.4f} {sm:>15.2f}')
if baseline_metrics:
    bm, _, _h = avg_predictions_per_film(bl_model, test_loader, bl_th)
    print(f'{"Baseline v1 (frozen)":<25} {baseline_metrics["macro_f1"]:>10.4f} {baseline_metrics["micro_f1"]:>10.4f} {baseline_metrics["hamming_loss"]:>10.4f} {bm:>15.2f}')
print(f'{"Fine-tuned v2":<25} {ft_metrics["macro_f1"]:>10.4f} {ft_metrics["micro_f1"]:>10.4f} {ft_metrics["hamming_loss"]:>10.4f} {mean_preds:>15.2f}')

## 5. Gorsel Ornekler — Tahmin vs Gercek

In [ ]:
test_df = pd.read_csv(TEST_CSV, dtype={'tmdb_id': str})
sample_indices = np.random.default_rng(0).choice(len(test_ds), size=12, replace=False)

ft_model.eval()
thresh_t = torch.tensor(ft_thresholds, device=DEVICE)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, idx in zip(axes.flat, sample_indices):
    tmdb_id = test_df.iloc[idx]['tmdb_id']
    img_pil = Image.open(POSTERS_DIR / f'{tmdb_id}.jpg').convert('RGB')

    img_tensor, true_label = test_ds[idx]
    with torch.no_grad():
        probs = torch.sigmoid(ft_model(img_tensor.unsqueeze(0).to(DEVICE)))[0]

    pred_labels = [g for g, p, t in zip(mlb.classes_, probs.cpu(), ft_thresholds) if p > t]
    true_labels = [g for g, v in zip(mlb.classes_, true_label) if v > 0.5]

    match = set(pred_labels) == set(true_labels)
    border_color = '#2ecc71' if match else '#e74c3c'

    ax.imshow(img_pil)
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    ax.set_title(
        f'Gercek: {", ".join(true_labels)}\nTahmin: {", ".join(pred_labels) or "-"}',
        fontsize=7
    )
    ax.axis('off')

plt.suptitle('Fine-tuned v2 — Tahminler (yesil cerceve = tam eslesme)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Sonuc Degerlendirmesi

In [ ]:
print('=' * 55)
print('BASARI KRITERLERI')
print('=' * 55)

# Kriter 1: v1 baseline'dan iyi mi?
bl_f1 = baseline_metrics['macro_f1'] if baseline_metrics else 0.3832
if ft_metrics['macro_f1'] > bl_f1:
    print(f'[OK] Macro F1 ({ft_metrics["macro_f1"]:.4f}) > Baseline v1 ({bl_f1:.4f})')
else:
    print(f'[!!] Macro F1 ({ft_metrics["macro_f1"]:.4f}) <= Baseline v1 ({bl_f1:.4f})')

# Kriter 2: Ortalama tahmin 1.5-3 arasi
if 1.5 <= mean_preds <= 3.0:
    print(f'[OK] Ort. tahmin/film ({mean_preds:.2f}) hedef araliginda (1.5-3.0)')
else:
    print(f'[!!] Ort. tahmin/film ({mean_preds:.2f}) hedef araliginin disinda')

# Kriter 3: History ve Documentary F1 yukseldi mi?
if scratch_metrics:
    hist_idx = list(mlb.classes_).index('History')
    doc_idx  = list(mlb.classes_).index('Documentary')
    hist_improvement = ft_metrics['per_class_f1'][hist_idx] - scratch_metrics['per_class_f1'][hist_idx]
    doc_improvement  = ft_metrics['per_class_f1'][doc_idx]  - scratch_metrics['per_class_f1'][doc_idx]
    print(f'[{'OK' if hist_improvement > 0 else '!!'}] History F1: {scratch_metrics["per_class_f1"][hist_idx]:.4f} -> {ft_metrics["per_class_f1"][hist_idx]:.4f} ({hist_improvement:+.4f})')
    print(f'[{'OK' if doc_improvement > 0 else '!!'}] Documentary F1: {scratch_metrics["per_class_f1"][doc_idx]:.4f} -> {ft_metrics["per_class_f1"][doc_idx]:.4f} ({doc_improvement:+.4f})')

print('=' * 55)